# K-Nearest Neighbors Regressor – Practice Skeleton

**Short name (GitHub):** `KNN_Regress`  
**Lab source:** Codecademy *K-Nearest Neighbor Regressor* (movies from-scratch + weighted average + `KNeighborsRegressor`)  
**Language:** Python (NumPy + pandas + Matplotlib + scikit-learn)

Use this notebook to practice. Open **`KNN_Regress_Solution.ipynb`** only after you attempt each exercise.  
Companion files: `KNN_Regress_Cheatsheet.docx`, `KNN_Regress_Reusable_Template.ipynb`, `knn_regress_flowchart.png`, `KNN_Regress.py`.

### Learning objectives
- Reuse Euclidean distance in 2-D and *n*-D
- Min-max normalize so budget (dollars) does not drown year / runtime
- Predict a **real-valued** IMDb-style rating as the mean of the *k* nearest ratings
- Upgrade the mean to an **inverse-distance weighted** average
- Fit `KNeighborsRegressor` (`weights="uniform"` vs `"distance"`)
- Sweep *k* with MAE / RMSE / R² (bias–variance for a continuous target)
- Alternate implementations (broadcast NumPy, Manhattan, `NearestNeighbors`)
- Extra practice: house prices and used-car prices
- Monte-Carlo: *k*, rating noise, sample size, extra noise features
- Rewrite the same forecast for an analyst, a product exec, a viewer, a non-specialist

### Data files
- `data/knn_movies_ratings.csv` — 80 films (duration, year, budget, rating)
- `data/knn_houses.csv` — 220 sales (sqft, year_built, baths → price_k)
- `data/knn_usedcars.csv` — 180 listings (mileage_k, age, engine_l → price_k)

### Flowchart
Open `knn_regress_flowchart.png` while you work.

### Classifier vs regressor (one sentence)
Classifier: majority **vote**. Regressor: average (or weighted average) of neighbor **numbers**.


## Inline cheat-sheet (keep this cell visible)

See also **`KNN_Regress_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Euclidean | $d(a,b)=\sqrt{\sum_j (a_j-b_j)^2}$ |
| Manhattan | $d_1(a,b)=\sum_j \|a_j-b_j\|$ |
| Min-max | $x'=(x-x_{\min})/(x_{\max}-x_{\min})$ |
| Uniform | $\hat{y}=\frac{1}{k}\sum_{i\in N_k} r_i$ |
| Weighted | $\hat{y}=\dfrac{\sum_{i\in N_k} r_i/d_i}{\sum_{i\in N_k} 1/d_i}$ |
| Guard | if $d_i=0$, that neighbor *is* the query — return its rating |
| Overfit | $k$ too small → one outlier rating owns $\hat{y}$ |
| Underfit | $k$ too large → $\hat{y}\approx$ global mean |
| sklearn | `KNeighborsRegressor(n_neighbors=k, weights="distance")` |
| Metrics | MAE, RMSE, $R^2$ — **not** accuracy |
| Split | `train_test_split(..., test_size=0.25, random_state=7)` |
| Scale | fit `MinMaxScaler` on **train** only |
| Cold start | a query with no nearby rows has nothing useful to average |

**Flow:** features → scale → split → distance → $k$ neighbors → average / weight → sweep $k$ → simulate.


## 0. Packages


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Distance between points (2-D)

Two movies as `[runtime_minutes, release_year]`.

$$
d = \sqrt{(L_1-L_2)^2 + (Y_1-Y_2)^2}
$$

### Task 1.1 — `distance_2d`
Write `distance_2d(movie1, movie2)` that returns the Euclidean distance. Use `** 0.5` or `math.sqrt`.


In [ ]:
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

def distance_2d(movie1, movie2):
    # YOUR CODE HERE
    pass

print(distance_2d(star_wars, raiders))
print(distance_2d(star_wars, mean_girls))
# Which film is closer to Star Wars?


## 2. Distance in *n* dimensions

$$
d(A,B)=\sqrt{\sum_{j=1}^{n}(A_j-B_j)^2}
$$

### Task 2.1 — generalize with a loop


In [ ]:
star_wars_3 = [125, 1977, 11_000_000]
raiders_3 = [115, 1981, 18_000_000]
mean_girls_3 = [97, 2004, 17_000_000]

def distance(a, b):
    """Euclidean distance for lists / 1-D arrays of any length."""
    # YOUR CODE HERE
    pass

print(distance(star_wars_3, raiders_3))
print(distance(star_wars_3, mean_girls_3))


### Task 2.2 — NumPy alternate (no Python loop)

`np.sqrt(np.sum((np.asarray(a)-np.asarray(b))**2))` or `np.linalg.norm`.


In [ ]:
def distance_np(a, b):
    # YOUR CODE HERE
    pass

print(distance_np(star_wars_3, raiders_3))
print("match loop?", np.isclose(distance(star_wars_3, raiders_3), distance_np(star_wars_3, raiders_3)))


## 3. Min-max normalization

Budget is dollars; year spans ~80. Without scaling, budget owns the distance.

$$
x' = \frac{x - x_{\min}}{x_{\max}-x_{\min}}
$$

A 2018 query against a training max of 2016 will have a scaled year **greater than 1**. That is expected — do not clip.

### Task 3.1 — `min_max_normalize(lst)`


In [ ]:
release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0,
                 1950.0, 1975.0, 1960.0, 2017.0, 1937.0]

def min_max_normalize(lst):
    # YOUR CODE HERE
    pass

print(min_max_normalize(release_dates))
# What does 1897 become? Why is it near 0?


## 4. Movie data + uniform regressor

### Task 4.1 — load `data/knn_movies_ratings.csv`
Build `movie_dataset` as `{title: [duration, year, budget]}` and `movie_ratings` as `{title: rating}`.


In [ ]:
# YOUR CODE HERE
movies = None
movie_dataset = {}
movie_ratings = {}

print("n movies:", len(movie_dataset))
print("Life of Pi features:", movie_dataset.get("Life of Pi"))
print("Life of Pi rating:", movie_ratings.get("Life of Pi"))


### Task 4.2 — column-wise min-max
Fit mins/maxs on the training dict, apply to every title and to any new query.


In [ ]:
def fit_minmax(dataset):
    """Return (mins, maxs) arrays of shape (n_features,)."""
    # YOUR CODE HERE
    pass

def apply_minmax(point, mins, maxs):
    # YOUR CODE HERE
    pass

def normalize_dataset(dataset):
    # YOUR CODE HERE
    pass

mins = maxs = None
movie_dataset_n = {}
print("mins:", mins)
print("maxs:", maxs)
print("Life of Pi scaled:", movie_dataset_n.get("Life of Pi"))


### Task 4.3 — `predict(unknown, dataset, movie_ratings, k)`  (uniform)

1. For every title compute `[distance, title]`.
2. Sort ascending.
3. Take the first *k*.
4. Average their ratings.
5. Return that mean.

This is the Codecademy checkpoint: *sum the ratings, divide by k*.


In [ ]:
def predict(unknown, dataset, movie_ratings, k):
    # YOUR CODE HERE
    pass


### Task 4.4 — predict *Incredibles 2*

Raw features used in this lab: runtime **118**, year **2018**, budget **200_000_000**.  
Scale with the **training** mins/maxs (year will be slightly above 1). Use *k* = 5. Print the uniform forecast.


In [ ]:
inc_raw = [118, 2018, 200_000_000]
# Scale, then call predict(..., k=5). Print ŷ.
# YOUR CODE HERE


## 5. Weighted regression

Closer neighbors should count more. With distances $d_i$ and ratings $r_i$:

$$
\hat{y} = \frac{\sum_i r_i / d_i}{\sum_i 1 / d_i}
$$

Toy check from the lesson (ratings 5.0, 6.8, 9.0 and distances 3.2, 11.5, 1.1): uniform mean ≈ 6.93, weighted ≈ 7.90.

### Task 5.1 — `predict_weighted`
Guard $d_i=0$ with a tiny `eps` (or return that neighbor's rating immediately).


In [ ]:
def predict_weighted(unknown, dataset, movie_ratings, k, eps=1e-12):
    # YOUR CODE HERE
    pass

# toy table from the lesson
toy_r = [5.0, 6.8, 9.0]
toy_d = [3.2, 11.5, 1.1]
# compute uniform and weighted by hand here to confirm 6.93 vs ~7.90
# YOUR CODE HERE


### Task 5.2 — same Incredibles 2 query, weighted, *k* = 5
How did the forecast move versus the uniform mean?


In [ ]:
# YOUR CODE HERE


## 6. Alternate neighbor search

### Task 6.1 — broadcast all pairwise distances
Stack the scaled dataset into `X` of shape `(m, n)` and compute `np.linalg.norm(X - unknown, axis=1)`.


In [ ]:
def predict_np(unknown, X, y, k, weighted=False, eps=1e-12):
    """unknown: (n,), X: (m,n), y: (m,) ratings."""
    # YOUR CODE HERE
    pass


### Task 6.2 — Manhattan distance variant
Replace the Euclidean norm with `np.abs(X - unknown).sum(axis=1)`. Compare the two *k* = 5 forecasts.


In [ ]:
def predict_manhattan(unknown, X, y, k):
    # YOUR CODE HERE
    pass


### Task 6.3 — `NearestNeighbors` alternate
`NearestNeighbors(n_neighbors=k).fit(X).kneighbors(unknown)` returns distances and indices. Average `y[idx]` yourself.


In [ ]:
# YOUR CODE HERE


## 7. scikit-learn `KNeighborsRegressor`

### Task 7.1
Create `regressor = KNeighborsRegressor(n_neighbors=5, weights="distance")`.  
Fit on the **scaled** feature matrix and the rating vector. Predict the three queries below and print the array.

- Incredibles 2 — the scaled vector you already computed
- A short cheap 2005 comedy you invent (scale it the same way)
- Life of Pi's own scaled vector (should land very near 7.9 — it is in the training set)


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
# YOUR CODE HERE
regressor = None


## 8. Sweep *k* and graph

### Task 8.1
`train_test_split` the scaled movies 75/25 (`random_state=7`). For `k` in 1…20 fit both `uniform` and `distance` regressors. Store validation MAE. Print the *k* that wins for each weighting.


In [ ]:
k_list = list(range(1, 21))
mae_uniform = []
mae_distance = []
# YOUR CODE HERE
print("best uniform k, MAE:", None, None)
print("best distance k, MAE:", None, None)


### Task 8.2 — plot
Two lines, x = `k_list`, y = MAE. Labels `"k"`, `"Validation MAE"`, title `"Movie rating KNN"`.


In [ ]:
# YOUR CODE HERE
plt.xlabel("k")
plt.ylabel("Validation MAE")
plt.title("Movie rating KNN")
plt.show()


## 9. More practice

### Task 9.1 — houses (`data/knn_houses.csv`)
Features `sqft`, `year_built`, `baths`; target `price_k`. Scale on train only, split 80/20, sweep *k* = 1…30, report best MAE and that *k*. Predict a 1,800 sqft / 2005 / 2-bath house.


In [ ]:
# YOUR CODE HERE


### Task 9.2 — used cars (`data/knn_usedcars.csv`)
Features `mileage_k`, `age`, `engine_l`; target `price_k`. Same protocol. Predict a 7-year-old 2.0 L car with 72k miles.


In [ ]:
# YOUR CODE HERE


## 10. Simulation (edit the box, re-run)

Change any of `K_FIXED`, `NOISE`, `TRAIN_FRAC`, `N_EXTRA`, `WEIGHTS` and re-run the cell.  
MAE is always scored against the **clean** hold-out ratings.


In [ ]:
# ----- editable -----
K_FIXED = 5            # neighbors
NOISE = 0.00           # sigma added to TRAIN ratings only
TRAIN_FRAC = 0.75
N_EXTRA = 0            # useless N(0,1) columns
WEIGHTS = "uniform"    # or "distance"
N_REPS = 12
SEED0 = 0
# --------------------

movies_s = pd.read_csv("data/knn_movies_ratings.csv")
X0 = movies_s[["duration", "year", "budget"]].to_numpy(float)
y0 = movies_s["rating"].to_numpy(float)
sc0 = MinMaxScaler().fit(X0)
X0s = sc0.transform(X0)

def one_run(seed):
    rng = np.random.default_rng(seed)
    X = X0s.copy()
    y_tr_src = y0 + rng.normal(0, NOISE, size=y0.shape)
    if N_EXTRA > 0:
        extra = rng.normal(0, 1, size=(len(X), N_EXTRA))
        X = np.hstack([X, extra])
        mn, mx = X.min(0), X.max(0)
        X = (X - mn) / np.where(mx - mn == 0, 1.0, mx - mn)
    n_tr = max(K_FIXED + 2, int(TRAIN_FRAC * len(X)))
    idx = rng.permutation(len(X))
    tr, va = idx[:n_tr], idx[n_tr:]
    if len(va) < 5:
        va, tr = idx[-8:], idx[:-8]
    k = min(K_FIXED, len(tr))
    model = KNeighborsRegressor(n_neighbors=k, weights=WEIGHTS)
    model.fit(X[tr], y_tr_src[tr])
    return mean_absolute_error(y0[va], model.predict(X[va]))

maes = [one_run(SEED0 + i) for i in range(N_REPS)]
print(f"mean MAE={np.mean(maes):.3f}  sd={np.std(maes):.3f}  "
      f"k={K_FIXED} noise={NOISE} frac={TRAIN_FRAC} extra={N_EXTRA} w={WEIGHTS}")


## 11. Audience rewrite

Using the attached audience notes (data literacy, subject knowledge, experts / technicians / executives / nonspecialists), write four short paragraphs for the same Incredibles 2 forecast and the hold-out MAE. Keep formulas out of the last two.


In [ ]:
# Write your four paragraphs as strings, then print them.
analyst = """..."""
product_exec = """..."""
viewer = """..."""
friend = """..."""
for label, text in [("analyst", analyst), ("exec", product_exec),
                    ("viewer", viewer), ("nonspecialist", friend)]:
    print("====", label, "====")
    print(text)
    print()


## Done

Compare with `KNN_Regress_Solution.ipynb`. Reuse the pattern on a new numeric table via `KNN_Regress_Reusable_Template.ipynb`.
